# RAG Minecraft

## Configuration Gemini API

In [42]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import format_document
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import Chroma
import shutil
import os
import csv
import requests
from urllib.parse import unquote
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
import cloudscraper
import time


In [43]:
from api_config import configure_google_api_key

GOOGLE_API_KEY = configure_google_api_key()

## Scrapping

Congig sources

In [44]:
WIKI_PAGES = [
    "Minecraft"
]

FANDOM_URLS = [
    "https://minecraft.fandom.com/fr/wiki/Survie",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif",
    "https://minecraft.fandom.com/fr/wiki/Hardcore",
    "https://minecraft.fandom.com/fr/wiki/Fabrication",
    "https://minecraft.fandom.com/fr/wiki/Cuisson",
    "https://minecraft.fandom.com/fr/wiki/Alchimie",
    "https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether",
    "https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon",
    "https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures"
]

In [45]:
scraper = cloudscraper.create_scraper()

In [46]:
def write_csv(file_name, paragraphs):
    with open(file_name, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["text"])
        for p in paragraphs:
            writer.writerow([p])

Wikipedia

In [47]:
def scrape_wikipedia(title: str):

    url = "https://fr.wikipedia.org/w/api.php"

    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
        "redirects": True
    }

    r = requests.get(url, params=params, headers={"User-Agent": "Mozilla/5.0"})
    data = r.json()

    html = data["parse"]["text"]["*"]
    soup = BeautifulSoup(html, "html.parser")

    texts = []
    capture = False

    for tag in soup.find_all(["h2", "p"]):

        t = tag.get_text(" ", strip=True)

        if tag.name == "h2" and "Références" in t:
            break

        if tag.name == "h2":
            capture = True
            continue

        if capture and t:
            texts.append(t)

    write_csv("files/wikipedia.csv", texts)

    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": f"wikipedia:{title}"}
    )

Fandom

In [48]:
def scrape_fandom(url: str):

    headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9",
    "Referer": "https://www.google.com/"
    }

    r = scraper.get(url, headers=headers)

    soup = BeautifulSoup(r.text, "html.parser")

    content = soup.find("div", {"class": "mw-parser-output"})
    if not content:
        return None

    texts = []

    for tag in content.find_all(["p", "h2", "h3", "li"]):

        t = tag.get_text(" ", strip=True)

        if not t:
            continue

        if len(t) < 20:
            continue

        if "Voir aussi" in t or "Notes et références" in t:
            break

        texts.append(t)

    paragraphs = [p.strip() for p in texts if p.strip()]
    page_name = url.split("/wiki/")[-1]
    page_name = unquote(page_name)
    file_name = f"files/{page_name}.csv"

    write_csv(file_name, paragraphs)


    return Document(
        page_content="\n\n".join(texts),
        metadata={"source": url}
    )

Build dataset

In [49]:
docs = []

# Wikipedia
for page in WIKI_PAGES:
    try:
        docs.append(scrape_wikipedia(page))
        print("WIKI OK:", page)
    except Exception as e:
        print("WIKI ERROR:", page, e)

# Fandom
for url in FANDOM_URLS:
    try:
        doc = scrape_fandom(url)
        if doc:
            docs.append(doc)
            print("FANDOM OK:", url)
    except Exception as e:
        print("FANDOM ERROR:", url, e)

print("\nTOTAL DOCS:", len(docs))

WIKI OK: Minecraft
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atif
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Hardcore
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Fabrication
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cuisson
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Alchimie
FANDOM OK: https://minecraft.fandom.com/fr/wiki/La_Foire_aux_Questions
FANDOM ERROR: https://minecraft.fandom.com/fr/wiki/Tutoriels/Choses_%C3%A0_ne_PAS_faire [Errno 2] No such file or directory: 'files/Tutoriels/Choses_à_ne_PAS_faire.csv'
FANDOM ERROR: https://minecraft.fandom.com/fr/wiki/Tutoriels/Survie_dans_le_Nether [Errno 2] No such file or directory: 'files/Tutoriels/Survie_dans_le_Nether.csv'
FANDOM ERROR: https://minecraft.fandom.com/fr/wiki/Tutoriels/L%27End_et_l%27Ender_Dragon [Errno 2] No such file or directory: "files/Tutoriels/L'End_et_l'Ender_Dragon.csv"
FANDOM OK: https://minecraft.fandom.com/fr/wiki/Cr%C3%A9atures

TOTAL DOCS: 8


Chunking

In [50]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=150
)

split_docs = splitter.split_documents(docs)

print("CHUNKS:", len(split_docs))

CHUNKS: 68


Enrchich

In [51]:
def enrich_with_source(docs):
    enriched = []

    for d in docs:
        src = d.metadata.get("source", "unknown")

        source_type = "WIKIPEDIA" if "wikipedia" in src else "FANDOM"

        d.page_content = (
            f"SOURCE: {src}\n"
            f"SOURCE_TYPE: {source_type}\n\n"
            f"{d.page_content}"
        )

        enriched.append(d)

    return enriched

split_docs = enrich_with_source(split_docs)

## Initialize embedding model + Retriever

In [52]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document"
)

#### Store data using chroma

In [53]:
import shutil
import os
path = "./chroma_minecraft_db"

if os.path.exists(path):
    try:
        shutil.rmtree(path)
    except PermissionError:
        print("dossier encore utilisé → restart kernel puis relance")

dossier encore utilisé → restart kernel puis relance


In [54]:
persist_dir = "./chroma_minecraft_db"

# 1. Création + stockage (UNE SEULE FOIS)
vectorstore = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)
batch_size = 10

for i in range(0, len(split_docs), batch_size):
    batch = split_docs[i:i+batch_size]
    vectorstore.add_documents(batch)
    time.sleep(2)

# 2. Reload propre (pour usage RAG)
vectorstore_disk = Chroma(
    persist_directory=persist_dir,
    embedding_function=gemini_embeddings
)

# 3. Retriever FINAL utilisé par le RAG
retriever = vectorstore_disk.as_retriever(
    search_kwargs={"k": 6}
)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

#### Create a retriever using Chroma

In [55]:
print(retriever.invoke("Minecraft")[0].page_content[:500])

def retrieve_documents(retriever, query):
    if hasattr(retriever, "invoke"):
        return retriever.invoke(query)
    if hasattr(retriever, "get_relevant_documents"):
        return retriever.get_relevant_documents(query)
    raise AttributeError("Le retriever ne supporte ni invoke ni get_relevant_documents.")

Mise à jour du monde

Mise à jour des Boss

Mise à jour de l'Ender

Mise à jour de la découverte

Mise à jour du mieux ensemble

Mise à jour aquatique 1

Mise à jour aquatique 2

Mise à jour des fêtes

Mise à jour Village & Pillage

Mise à jour Abeilles bourdonnantes

Mise à jour du Nether

Historique des versions

PlayStation PSVita PS3 PS4

Minecraft: Education Edition

Codes de mise en forme

Format de carte .schematic

Format de fichier Région

Format de fichier Anvil

Format de carte Alpha



## Generator

#### Initialize Generator

In [56]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

#### Create prompt templates

In [57]:
# Prompt template to query Gemini
llm_prompt_template = """You are an assistant for question-answering tasks.
Use the following context to answer the question.
If you don't know the answer, just say that you don't know.
Use five sentences maximum and keep the answer concise.\n
Question: {question} \nContext: {context} \nAnswer:"""

llm_prompt = PromptTemplate.from_template(llm_prompt_template)

print(llm_prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} template="You are an assistant for question-answering tasks.\nUse the following context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse five sentences maximum and keep the answer concise.\n\nQuestion: {question} \nContext: {context} \nAnswer:"


#### Create a stuff documents chain

In [58]:
# Combine data from documents to readable string format.
def format_docs(docs):
    formatted = []

    for doc in docs:
        src = doc.metadata.get("source", "unknown")
        formatted.append(f"[{src}]\n{doc.page_content[:1000]}")

    return "\n\n".join(formatted)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | llm_prompt
    | llm
    | StrOutputParser()
)

### Prompt the model

In [ ]:
Markdown(rag_chain.invoke("Quels sont les principaux modes de jeu dans Minecraft ?"))

D'après les textes fournis, les principaux modes de jeu de Minecraft sont :

* **Le mode Créatif**, axé sur la construction libre avec des ressources illimitées et sans aspect de survie.
* **Le mode Survie**, dans lequel les joueurs doivent récolter des ressources, combattre des monstres, construire et gérer leur santé et leur faim.
* **Le mode Hardcore** (« Extrême »), une variante du mode Survie où la difficulté est verrouillée en mode difficile et où la mort est définitive. 
* Le **mode Spectateur** est également mentionné comme l'état dans lequel le joueur réapparaît après être mort en mode Hardcore.

In [60]:
Markdown(rag_chain.invoke("Comment survivre dans le Nether dans Minecraft ?"))

Pour survivre face aux créatures du Nether, il faut savoir qu'elles sont immunisées contre le feu et la lave et qu'elles apparaissent indépendamment du niveau de luminosité. 

Faites particulièrement attention aux Ghasts, qui lancent des boules de feu explosives capables de détruire la netherrack et de vous faire tomber dans la lave. Pour vous défendre, vous pouvez dévier la trajectoire de ces boules de feu et les renvoyer directement sur le Ghast.

In [59]:
Markdown(rag_chain.invoke("C'est quoi l'alchimie dans Minecraft?"))

L'alchimie (ou distillation) est la science permettant de créer des potions, des potions jetables et des potions persistantes dans Minecraft. Pour la pratiquer, le joueur doit fabriquer un alambic et utiliser sa grille d'alchimie pour distiller divers ingrédients dans des fioles d'eau. La verrue du Nether est l'ingrédient de base principal, indispensable pour concevoir la majorité des potions. Il est également possible d'utiliser des modificateurs pour altérer les propriétés ou changer les effets des potions créées. Enfin, l'ajout de poudre à canon ou de souffle de dragon permet de transformer ces potions en versions jetables ou persistantes.

In [62]:
Markdown(rag_chain.invoke("Que sont les monstres dans Minecraft?"))

Dans Minecraft, les monstres font partie des créatures (ou « mobs »), qui sont des entités se déplaçant par elles-mêmes avec des comportements déterminés. Les créatures hostiles, qui correspondent aux monstres, se caractérisent par le fait qu'elles attaquent le joueur à vue. Le jeu comprend divers exemples de ces créatures, tels que le zombie, le squelette, le Creeper, l'Enderman, le noyé ou la sorcière. Enfin, certains monstres sont des boss, c'est-à-dire des créatures hostiles spéciales dotées de très nombreux points de vie et d'attaques complexes.

In [65]:
Markdown(rag_chain.invoke("Liste les boss dans Minecraft?"))

Le contexte indique qu'il existe trois boss dans Minecraft, mais il ne donne pas la liste de leurs noms. Par conséquent, je ne sais pas quels sont précisément ces boss.

In [66]:
Markdown(rag_chain.invoke("C'est quoi le warden dans Minecraft?"))

Je ne sais pas, car le contexte fourni ne contient aucune information sur le warden dans Minecraft.

In [67]:
Markdown(rag_chain.invoke("Quant a été créé Minecraft?"))

Minecraft a été créé le 10 mai 2009 par Notch, de son vrai nom Markus Persson.